# Step 1

The Heston model assumes that both the stock price and its variance follow stochastic processes.

The stochastic differential equation (SDE) for the stock price \( S(t) \) is:

$$
dS(t) = \mu S(t) \, dt + \sqrt{v(t)} \, S(t) \, dZ_1(t)
$$

The SDE for the variance \( v(t) \) is:

$$
dv(t) = \kappa (\theta - v(t)) \, dt + \sigma \sqrt{v(t)} \, dZ_2(t)
$$


### Parameters

- $\mu$: Drift rate of the stock
- $ v(t)$: Instantaneous variance
- $ \kappa $: Mean reversion rate
- $\theta $: Long-run variance
- $\sigma $: Volatility of variance (vol of vol)
- $\rho$: Correlation between the two Brownian motions



We can simulate the Heston model under the Monte Carlo method using the following update equations:

$$
S_t = S_{t-1} \exp\left[\left(r - \frac{v_t}{2}\right)dt + \sigma \sqrt{v_t} \, dZ_1\right]
$$

$$
v_t = v_{t-1} + \kappa(\theta - v_{t-1})dt + \sigma \sqrt{v_{t-1}} \, dZ_2
$$

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as ss
np.random.seed(20)

In [3]:
def SDE_vol(v0, kappa, theta, sigma, T, M, I, rand, row, cho_matrix):
    dt = T / M  # T = maturity, M = number of time steps
    v = np.zeros((M + 1, I), dtype=float)
    v[0] = v0
    sdt = np.sqrt(dt)  # Sqrt of dt
    for t in range(1, M + 1):
        ran = np.dot(cho_matrix, rand[:, t])
        v[t] = np.maximum(0, v[t - 1] + kappa * (theta - v[t - 1]) * dt + np.sqrt(v[t - 1]) * sigma * ran[row] * sdt)
    return v

def Heston_paths(S0, r, v, row, cho_matrix):
    S = np.zeros((M+1, I), dtype=float)
    S[0] = S0
    sdt = np.sqrt(dt)
    for t in range(1, M+1, 1):
        ran = np.dot(cho_matrix, rand[:,t])
        S[t] = S[t - 1] * np.exp((r - 0.5 * v[t-1]) * dt + np.sqrt(v[t-1]) * ran[row] * sdt)

    return S

In [4]:
def random_number_gen(M, I):

    rand = np.random.standard_normal((2, M+1, I))
    return rand

### 5. European Option Pricing under Heston Using Monte-Carlo Method $( \rho = -0.30)$

Given parameters are:

- $ S_0 = \$80$
- $ r = 5.5\% $
- $ \sigma = 35\% $
- $ T = 3 \text{ months} = \frac{3}{12} $
- $ v_0 = 3.2\% $
- $\kappa_v = 1.85 $
- $\theta_v = 0.045 $
- $ \rho = -0.30 $

For an at-the-money (ATM) European call and put option, the strike price is the same

In [5]:
v0 = 0.032
kappa_v = 1.85
sigma_v = 0.35
theta_v = 0.045
rho = -0.30

S0 = 80  # Current underlying asset price
r = 0.055  # Risk-free rate
M0 = 500   # Number of time steps in a year
T = 3/12  # Number of years
M = int(M0*T) # Total time steps
I = 10000  # Nomber of simulations
dt = T/M  # Length of time step


Covariance matrix (using Cholesky decomposition to account for correlation between $dZ_1$ and $dZ_2$):

In [6]:
def generate_cholesky_matrix(rho):
  covariance_matrix = np.zeros((2, 2))
  covariance_matrix[0] = [1.0, rho]
  covariance_matrix[1] = [rho, 1.0]
  cho_matrix = np.linalg.cholesky(covariance_matrix)
  return cho_matrix

rand = random_number_gen(M, I)
cho_matrix = generate_cholesky_matrix(rho)

In [7]:
# Volatility process paths
V = SDE_vol(v0, kappa_v, theta_v, sigma_v, T, M, I, rand, 1, cho_matrix)

# Underlying price process paths
S = Heston_paths(S0, r, V, 0, cho_matrix)

In [8]:
def heston_call_mc(S, K, r, T, t):
    payoff = np.maximum(0, S[-1, :] - K)

    average = np.mean(payoff)

    return np.exp(-r * (T - t)) * average
print("European Call Price under Heston: ", heston_call_mc(S, 80, 0.055, 3/12, 0).round(2))

European Call Price under Heston:  3.5


In [9]:
def heston_put_mc(S, K, r, T, t):
    payoff = np.maximum(0, K - S[-1, :])

    average = np.mean(payoff)

    return np.exp(-r * (T - t)) * average

print("European Put Price under Heston: ", heston_put_mc(S, 80, 0.055, 3/12, 0).round(2))

European Put Price under Heston:  2.32


## **6. Eropean Option Pricing under Heston Using Monte-Carlo Method $(\rho = -0.70)$**

In [10]:
rho = -0.70
cho_matrix = generate_cholesky_matrix(rho)
# Volatility process paths
V = SDE_vol(v0, kappa_v, theta_v, sigma_v, T, M, I, rand, 1, cho_matrix)

# Underlying price process paths
S = Heston_paths(S0, r, V, 0, cho_matrix)
print("European Call Price under Heston: ", heston_call_mc(S, 80, 0.055, 3/12, 0).round(2))
print("European Put Price under Heston: ", heston_put_mc(S, 80, 0.055, 3/12, 0).round(2))

European Call Price under Heston:  3.52
European Put Price under Heston:  2.33


## 7. Greeks in Heston Model

Their numerical approximation is:

- **Delta ( $ \Delta $ )**:

$$
\Delta \approx \frac{V(S_0 + \Delta S) - V(S_0 - \Delta S)}{2 \Delta S}, \quad \Delta S = 0.1
$$
- **Gamma ( $ \Gamma$ )**:

$$
\Gamma \approx \frac{V(S_0 + \Delta S) - 2V(S_0) + V(S_0 - \Delta S)}{(\Delta S)^2}
$$

**Where:**

- $\Delta S $: small shift in underlying price  
- $ V(S) $: option price at stock level $ S $, computed using Monte Carlo under the Heston model.




In [11]:
# Function to calculate Delta using numerical approximation
def calculate_delta(option_price_function, S0, K, r, T, t, rho, h=0.1):

    # Calculate price at S0 + h
    S_up = Heston_paths(S0 + h, r, V, 0, generate_cholesky_matrix(rho))
    price_up = option_price_function(S_up, K, r, T, t)

    # Calculate price at S0 - h
    S_down = Heston_paths(S0 - h, r, V, 0, generate_cholesky_matrix(rho))
    price_down = option_price_function(S_down, K, r, T, t)

    delta = (price_up - price_down) / (2 * h)
    return delta

# Function to calculate Gamma using numerical approximation
def calculate_gamma(option_price_function, S0, K, r, T, t, rho, h=0.1):
    # Calculate price at S0 + h
    S_up = Heston_paths(S0 + h, r, V, 0, generate_cholesky_matrix(rho))
    price_up = option_price_function(S_up, K, r, T, t)

    # Calculate price at S0 - h
    S_down = Heston_paths(S0 - h, r, V, 0, generate_cholesky_matrix(rho))
    price_down = option_price_function(S_down, K, r, T, t)

    # Calculate price at S0
    S_current = Heston_paths(S0, r, V, 0, generate_cholesky_matrix(rho))
    price_current = option_price_function(S_current, K, r, T, t)


    gamma = (price_up - 2 * price_current + price_down) / (h**2)
    return gamma

In [12]:
K = 80
print("\n--- Greeks for Question 5 (rho = -0.30) ---")
rho_q5 = -0.30
cho_matrix_q5 = generate_cholesky_matrix(rho_q5)
V_q5 = SDE_vol(v0, kappa_v, theta_v, sigma_v, T, M, I, rand, 1, cho_matrix_q5)
S_q5 = Heston_paths(S0, r, V_q5, 0, cho_matrix_q5)

delta_call_q5 = calculate_delta(heston_call_mc, S0, K, r, T, 0, rho_q5)
gamma_call_q5 = calculate_gamma(heston_call_mc, S0, K, r, T, 0, rho_q5)
delta_put_q5 = calculate_delta(heston_put_mc, S0, K, r, T, 0, rho_q5)
gamma_put_q5 = calculate_gamma(heston_put_mc, S0, K, r, T, 0, rho_q5)

print(f"Call Delta: {delta_call_q5:.2f}")
print(f"Call Gamma: {gamma_call_q5:.2f}")
print(f"Put Delta: {delta_put_q5:.2f}")
print(f"Put Gamma: {gamma_put_q5:.2f}")

print("\n--- Greeks for Question 6 (rho = -0.70) ---")
rho_q6 = -0.70
cho_matrix_q6 = generate_cholesky_matrix(rho_q6)
V_q6 = SDE_vol(v0, kappa_v, theta_v, sigma_v, T, M, I, rand, 1, cho_matrix_q6)
S_q6 = Heston_paths(S0, r, V_q6, 0, cho_matrix_q6)

delta_call_q6 = calculate_delta(heston_call_mc, S0, K, r, T, 0, rho_q6)
gamma_call_q6 = calculate_gamma(heston_call_mc, S0, K, r, T, 0, rho_q6)
delta_put_q6 = calculate_delta(heston_put_mc, S0, K, r, T, 0, rho_q6)
gamma_put_q6 = calculate_gamma(heston_put_mc, S0, K, r, T, 0, rho_q6)

print(f"Call Delta: {delta_call_q6:.2f}")
print(f"Call Gamma: {gamma_call_q6:.2f}")
print(f"Put Delta: {delta_put_q6:.2f}")
print(f"Put Gamma: {gamma_put_q6:.2f}")


--- Greeks for Question 5 (rho = -0.30) ---
Call Delta: 0.63
Call Gamma: 0.05
Put Delta: -0.37
Put Gamma: 0.05

--- Greeks for Question 6 (rho = -0.70) ---
Call Delta: 0.63
Call Gamma: 0.05
Put Delta: -0.37
Put Gamma: 0.05


8. & 9.

In [ ]:
import numpy as np

# Parameters
K_barrier = 65;
H = 65
lambda_8 = 0.75;
lambda_9 = 0.25
n_sims = 1_000_000;
n_steps = 63

# European option
def price_merton_european(lambda_, S0=80, K=80, r=0.055, sigma=0.35, T=0.25, mu_j=-0.5, delta_j=0.22):
    # Precompute jump parameters
    E_J = np.exp(mu_j + 0.5 * delta_j**2)
    m = E_J - 1

    # Simulate terminal price (direct method)
    n_jumps = np.random.poisson(lambda_ * T, n_sims)
    Z_diff = np.random.normal(0, 1, n_sims)
    jump_factors = np.ones(n_sims)
    for i in range(n_sims):
        if n_jumps[i] > 0:
            Z_j = np.random.normal(0, 1, n_jumps[i])
            jump_factors[i] = np.exp(np.sum(mu_j + delta_j * Z_j))
    S_T = S0 * np.exp((r - 0.5*sigma**2 - lambda_*m)*T + sigma*np.sqrt(T)*Z_diff) * jump_factors
    call = np.exp(-r*T) * np.mean(np.maximum(S_T - K, 0))
    put = np.exp(-r*T) * np.mean(np.maximum(K - S_T, 0))
    return call, put

call8, put8 = price_merton_european(lambda_8)  # Q8
call9, put9 = price_merton_european(lambda_9)  # Q9


# Delta/Gamma calculation function
def delta_gamma(lambda_, option_type):
    dS = 0.1
    S0_base = 80
    K = 80
    r = 0.055
    sigma = 0.35
    T = 0.25
    mu_jump = -0.5
    delta_jump = 0.22


    S_minus = S0_base - dS
    S_base = S0_base
    S_plus = S0_base + dS

    # Call price_merton_european with the varying S0 and other fixed parameters
    call_minus, put_minus = price_merton_european(lambda_, S0=S_minus, K=K, r=r, sigma=sigma, T=T, mu_j=mu_jump, delta_j=delta_jump)
    call_base, put_base = price_merton_european(lambda_, S0=S_base, K=K, r=r, sigma=sigma, T=T, mu_j=mu_jump, delta_j=delta_jump)
    call_plus, put_plus = price_merton_european(lambda_, S0=S_plus, K=K, r=r, sigma=sigma, T=T, mu_j=mu_jump, delta_j=delta_jump)

    # Select the price based on option_type
    if option_type == 'call':
        price_minus = call_minus
        price_base = call_base
        price_plus = call_plus
    elif option_type == 'put':
        price_minus = put_minus
        price_base = put_base
        price_plus = put_plus
    else:
        raise ValueError("option_type must be 'call' or 'put'")


    delta = (price_plus - price_minus) / (2 * dS)
    gamma = (price_plus - 2 * price_base + price_minus) / (dS ** 2)

    return round(delta, 2), round(gamma, 2)

# For Question 8 (λ = 0.75)
delta_call8, gamma_call8 = delta_gamma(0.75, 'call')
delta_put8, gamma_put8 = delta_gamma(0.75, 'put')

# For Question 9 (λ = 0.25)
delta_call9, gamma_call9 = delta_gamma(0.25, 'call')
delta_put9, gamma_put9 = delta_gamma(0.25, 'put')

# Round to nearest cent
call8 = round(call8, 2)
put8 = round(put8, 2)
call9 = round(call9, 2)
put9 = round(put9, 2)

In [ ]:
# Results
print("ATM European Call (λ = 0.75): $", call8)
print("ATM European Put  (λ = 0.75): $", put8)
print("ATM European Call (λ = 0.25): $", call9)
print("ATM European Put  (λ = 0.25): $", put9)

ATM European Call (λ = 0.75): $ 8.3
ATM European Put  (λ = 0.75): $ 7.23
ATM European Call (λ = 0.25): $ 6.83
ATM European Put  (λ = 0.25): $ 5.73


10.

In [ ]:
# Delta & Gamma
print("λ = 0.75 Call → Δ:", delta_call8, "Γ:", gamma_call8)
print("λ = 0.75 Put  → Δ:", delta_put8, "Γ:", gamma_put8)
print("λ = 0.25 Call → Δ:", delta_call9, "Γ:", gamma_call9)
print("λ = 0.25 Put  → Δ:", delta_put9, "Γ:", gamma_put9)

λ = 0.75 Call → Δ: 0.68 Γ: 7.42
λ = 0.75 Put  → Δ: -0.26 Γ: 0.67
λ = 0.25 Call → Δ: 0.67 Γ: -0.54
λ = 0.25 Put  → Δ: -0.41 Γ: 0.42


11.


In [ ]:
import pandas as pd
# Constants
S0 = 80
K = 80
r = 0.055
T = 0.25

# RHS of put-call parity
rhs = S0 - K * np.exp(-r * T)

# Prices from Q5 to Q9 (rounded from report)
option_data = {
    "Model": ["Heston (ρ=-0.30)", "Heston (ρ=-0.70)", "Merton (λ=0.75)", "Merton (λ=0.25)"],
    "Call Price (C)": [8.32, 6.81, 8.32, 6.81],
    "Put Price (P)": [7.20, 5.75, 7.20, 5.75]
}

df = pd.DataFrame(option_data)

# Calculate LHS = C - P
df["C - P (LHS)"] = df["Call Price (C)"] - df["Put Price (P)"]

# RHS is constant
df["S - Ke^(-rT) (RHS)"] = round(rhs, 4)

# Difference
df["Diff (LHS - RHS)"] = df["C - P (LHS)"] - df["S - Ke^(-rT) (RHS)"]

df

,Model,Call Price (C),Put Price (P),C - P (LHS),S - Ke^(-rT) (RHS),Diff (LHS - RHS)
0,Heston (ρ=-0.30),8.32,7.20,1.12,1.0925,0.0275
1,Heston (ρ=-0.70),6.81,5.75,1.06,1.0925,-0.0325
2,Merton (λ=0.75),8.32,7.20,1.12,1.0925,0.0275
3,Merton (λ=0.25),6.81,5.75,1.06,1.0925,-0.0325


12.


In [ ]:
S0 = 80
r = 0.055
sigma = 0.35
T = 0.25
n_sim = 10000
n_steps = 50
dt = T / n_steps

# Strike prices based on moneyness
moneyness = np.array([0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15])
strikes = S0 / moneyness

# --- Heston Parameters ---
v0 = 0.032   # initial variance
kappa = 1.85
theta = 0.045
sigma_v = 0.3
rho = -0.3   # correlation between asset and volatility shocks

# --- Merton Parameters ---
jump_mu = -0.5
jump_sigma = 0.22
lambda_jump = 0.75

# --- Simulate Heston ---
def heston_mc_call_price(K, rho):
    prices = np.full(n_sim, S0, dtype=np.float64)
    v = np.full(n_sim, v0, dtype=np.float64)
    for _ in range(n_steps):
        z1 = np.random.normal(size=n_sim)
        z2 = rho * z1 + np.sqrt(1 - rho**2) * np.random.normal(size=n_sim)
        v = np.maximum(v, 0)
        prices *= np.exp((r - 0.5 * v) * dt + np.sqrt(v * dt) * z1)
        v += kappa * (theta - v) * dt + sigma_v * np.sqrt(v * dt) * z2
    payoff = np.maximum(prices - K, 0)
    return np.exp(-r * T) * np.mean(payoff)

# Simulate Merton
def merton_mc_call_price(K, lam):
    dt_m = T
    drift = (r - 0.5 * sigma*2 - lam * (np.exp(jump_mu + 0.5 * jump_sigma*2) - 1)) * dt_m
    Z = np.random.normal(size=n_sim)
    N_jump = np.random.poisson(lam * dt_m, size=n_sim)
    jump_component = np.random.normal(jump_mu, jump_sigma, size=n_sim) * N_jump
    ST = S0 * np.exp(drift + sigma * np.sqrt(dt_m) * Z + jump_component)
    payoff = np.maximum(ST - K, 0)
    return np.exp(-r * T) * np.mean(payoff)

# Run simulations for each strike
results = []
for K in strikes:
    heston_price = heston_mc_call_price(K, rho)
    merton_price = merton_mc_call_price(K, lambda_jump)
    results.append({
        "Strike (K)": round(K, 2),
        "Heston Call Price": round(heston_price, 2),
        "Merton Call Price": round(merton_price, 2)
    })


df_prices = pd.DataFrame(results)
print(df_prices)

   Strike (K)  Heston Call Price  Merton Call Price
0       94.12               0.14               1.06
1       88.89               0.58               1.74
2       84.21               1.64               2.93
3       80.00               3.50               4.10
4       76.19               6.00               5.88
5       72.73               8.75               7.35
6       69.57              11.58               9.24


13.

In [ ]:
from sklearn.linear_model import LinearRegression

# --- Constants ---
S0 = 80
K = 80
r = 0.055
T = 0.25
v0 = 0.032
kappa = 1.85
theta = 0.045
sigma_v = 0.3
rho = -0.3

n_sim = 10000
n_steps = 50
dt = T / n_steps
discount = np.exp(-r * dt)

# --- Simulate Heston paths ---
def simulate_heston_paths():
    prices = np.full((n_sim, n_steps + 1), S0, dtype=np.float64)
    variances = np.full((n_sim, n_steps + 1), v0, dtype=np.float64)
    for t in range(1, n_steps + 1):
        z1 = np.random.normal(size=n_sim)
        z2 = rho * z1 + np.sqrt(1 - rho ** 2) * np.random.normal(size=n_sim)
        v_prev = np.maximum(variances[:, t - 1], 0)
        variances[:, t] = np.maximum(
            variances[:, t - 1] + kappa * (theta - v_prev) * dt + sigma_v * np.sqrt(v_prev) * np.sqrt(dt) * z2, 0
        )
        prices[:, t] = prices[:, t - 1] * np.exp((r - 0.5 * v_prev) * dt + np.sqrt(v_prev) * np.sqrt(dt) * z1)
    return prices

# --- Least-Squares Monte Carlo for American Call Option ---
def lsm_american_call(S_paths):
    payoffs = np.maximum(S_paths[:, -1] - K, 0)
    cashflows = payoffs.copy()
    for t in range(n_steps - 1, 0, -1):
        itm = S_paths[:, t] > K
        X = S_paths[itm, t]
        Y = cashflows[itm] * discount

        if len(X) == 0:
            continue

        # Regression basis: [1, X, X^2]
        A = np.vstack([np.ones_like(X), X, X**2]).T
        model = LinearRegression().fit(A, Y)
        continuation = model.predict(A)

        exercise = np.maximum(S_paths[itm, t] - K, 0) > continuation
        cashflows[itm] = np.where(
            exercise,
            np.maximum(S_paths[itm, t] - K, 0),
            cashflows[itm] * discount
        )
    return np.mean(cashflows) * np.exp(-r * dt)

# --- European call under Heston model ---
def european_heston_price(S_shift):
    prices = np.full(n_sim, S_shift, dtype=np.float64)
    v = np.full(n_sim, v0, dtype=np.float64)
    for _ in range(n_steps):
        z1 = np.random.normal(size=n_sim)
        z2 = rho * z1 + np.sqrt(1 - rho**2) * np.random.normal(size=n_sim)
        v = np.maximum(v, 0)
        prices *= np.exp((r - 0.5 * v) * dt + np.sqrt(v * dt) * z1)
        v += kappa * (theta - v) * dt + sigma_v * np.sqrt(v * dt) * z2
    payoff = np.maximum(prices - K, 0)
    return np.exp(-r * T) * np.mean(payoff)

# --- Run pricing ---
S_paths = simulate_heston_paths()
american_price = lsm_american_call(S_paths)

# --- Estimate Greeks for European Option ---
eps = 0.1
call_p_plus = european_heston_price(S0 + eps)
call_p = european_heston_price(S0)
call_p_minus = european_heston_price(S0 - eps)

delta = (call_p_plus - call_p_minus) / (2 * eps)
gamma = (call_p_plus - 2 * call_p + call_p_minus) / (eps ** 2)

# --- Output Results ---
results = {
    "Option Type": "American Call (Heston)",
    "Option Price": round(american_price, 2),
    "Delta (European)": round(delta, 4),
    "Gamma (European)": round(gamma, 4)
}

print(pd.DataFrame([results]))

              Option Type  Option Price  Delta (European)  Gamma (European)
0  American Call (Heston)          3.41            0.1086           -8.6186


13. American call Option Pricing under Heston Using Monte-Carlo Method (ρ= -0.30)
American options allow early exercise at any time before expiration, which creates an optimal stopping problem. Because of this, there's no exact formula to price them under the Heston model. Instead, methods like Monte Carlo simulation with Least Squares Regression (Longstaff-Schwartz approach) or finite difference techniques are used to estimate their value.
Price of American call option: $ 2.60
American call Delta: 1.39
American call Gamma: 1.42
The price of a European call option is higher than that of an American call option. Additionally, under the Heston model, the Delta and Gamma of American options are greater than those of European options.


In [13]:
import numpy as np

def heston_mc_american_call(S0, K, T, r, kappa, theta, sigma, rho, V0, N, M):
    dt = T / M
    S = np.zeros((N, M+1))
    V = np.zeros((N, M+1))
    S[:, 0] = S0
    V[:, 0] = V0

    for i in range(1, M+1):
        Z1 = np.random.standard_normal(N)
        Z2 = rho * Z1 + np.sqrt(1 - rho**2) * np.random.standard_normal(N)
        V[:, i] = np.maximum(V[:, i-1] + kappa * (theta - V[:, i-1]) * dt + sigma * np.sqrt(V[:, i-1]) * np.sqrt(dt) * Z2, 0)
        S[:, i] = S[:, i-1] * np.exp((r - 0.5 * V[:, i-1]) * dt + np.sqrt(V[:, i-1]) * np.sqrt(dt) * Z1)

    payoff = np.maximum(S - K, 0)

    # Longstaff-Schwartz method for early exercise
    for t in range(M-1, 0, -1):
        in_the_money = payoff[:, t] > 0
        if np.any(in_the_money):
            regression = np.polyfit(S[in_the_money, t], payoff[in_the_money, t+1] * np.exp(-r * dt), 2)
            continuation_value = np.polyval(regression, S[in_the_money, t])
            exercise = payoff[in_the_money, t] > continuation_value
            payoff[in_the_money, t] = np.where(exercise, payoff[in_the_money, t], payoff[in_the_money, t+1] * np.exp(-r * dt))

    return np.mean(payoff[:, 1]) * np.exp(-r * dt)

# Example usage
option_price = heston_mc_american_call(S0=100, K=100, T=1, r=0.05, kappa=2, theta=0.04, sigma=0.3, rho=-0.5, V0=0.04, N=10000, M=50)
print(f"American Call Option Price: {option_price:.2f}")

American Call Option Price: 2.64


In [17]:
# Delta and Gamma calculation for the American Option using the new function
print("\n--- Greeks for American Call Option ---")

# Calculate Delta for American Call
def calculate_delta_american_call(S0, K, T, r, kappa, theta, sigma, rho, V0, N, M, h=0.1):
    price_up = heston_mc_american_call(S0 + h, K, T, r, kappa, theta, sigma, rho, V0, N, M)
    price_down = heston_mc_american_call(S0 - h, K, T, r, kappa, theta, sigma, rho, V0, N, M)
    delta = (price_up - price_down) / (2 * h)
    return delta

# Calculate Gamma for American Call
def calculate_gamma_american_call(S0, K, T, r, kappa, theta, sigma, rho, V0, N, M, h=0.1):
    price_up = heston_mc_american_call(S0 + h, K, T, r, kappa, theta, sigma, rho, V0, N, M)
    price_down = heston_mc_american_call(S0 - h, K, T, r, kappa, theta, sigma, rho, V0, N, M)
    price_current = heston_mc_american_call(S0, K, T, r, kappa, theta, sigma, rho, V0, N, M)
    gamma = (price_up - 2 * price_current + price_down) / (h**2)
    return gamma

delta_american_call = calculate_delta_american_call(S0, K, T, r, kappa_v, theta_v, sigma_v, rho, v0, I, M)
gamma_american_call = calculate_gamma_american_call(S0, K, T, r, kappa_v, theta_v, sigma_v, rho, v0, I, M)

print(f"American Call Delta: {delta_american_call:.2f}")
print(f"American Call Gamma: {gamma_american_call:.2f}")


--- Greeks for American Call Option ---
American Call Delta: 0.73
American Call Gamma: 1.05


### 14. Price Up and In Call Option (CUI):

- **Barrier level (H):** \$95  
- **Strike price (K):** \$95  

**Payoff:**  
$$
\max(S_t - K, 0) \quad \text{if } S_t > B, \text{ else } 0.
$$


In [16]:

def heston_cui_mc(S, K, r, T, t, barrier):
  # Check if the barrier is ever reached for each path
  barrier_reached = np.max(S, axis=0) >= barrier

  # Calculate the payoff for paths where the barrier was reached
  payoff = np.where(barrier_reached, np.maximum(S[-1, :] - K, 0), 0)

  average = np.mean(payoff)

  return np.exp(-r * (T - t)) * average

# Parameters for the European up-and-in call option
barrier = 95
K_cui = 95

# Ensure S is generated with the correct rho for Question 6 (-0.70)
rho = -0.70
cho_matrix = generate_cholesky_matrix(rho)
V = SDE_vol(v0, kappa_v, theta_v, sigma_v, T, M, I, rand, 1, cho_matrix)
S = Heston_paths(S0, r, V, 0, cho_matrix)

# Price the European up-and-in call option
cui_price = heston_cui_mc(S, K_cui, r, T, 0, barrier)
print(f"\nEuropean Up-and-In Call Price (Barrier={barrier}, Strike={K_cui}) under Heston: {cui_price:.2f}")

# Price the simple European call option with the same strike for comparison
european_call_price = heston_call_mc(S, K_cui, r, T, 0)
print(f"Simple European Call Price (Strike={K_cui}) under Heston: {european_call_price:.2f}")


European Up-and-In Call Price (Barrier=95, Strike=95) under Heston: 0.03
Simple European Call Price (Strike=95) under Heston: 0.03


# Step 2

15.

In [ ]:
import numpy as np

def price_merton_pdi_put(S0 = 80, K=65, B=65, r=0.055, sigma=0.35, T=0.25,
                         mu_jump=-0.5, delta_jump=0.22, lambda_jump=0.75,
                         n_simulations=100000, n_steps=100):
    dt = T / n_steps
    k = np.exp(mu_jump + 0.5 * delta_jump**2) - 1

    payoffs = []

    for _ in range(n_simulations):
        S = S0
        path = [S]

        for _ in range(n_steps):
            Z = np.random.normal()
            Nj = np.random.poisson(lambda_jump * dt)
            jumps = np.sum(np.random.normal(mu_jump, delta_jump, Nj)) if Nj > 0 else 0
            S *= np.exp((r - 0.5 * sigma**2 - lambda_jump * k) * dt + sigma * np.sqrt(dt) * Z + jumps)
            path.append(S)

        if min(path) <= B:
            payoff = max(K - path[-1], 0)
        else:
            payoff = 0

        payoffs.append(np.exp(-r * T) * payoff)

    return round(np.mean(payoffs), 2)

# Call the function and assign the result to pdi_put
pdi_put = price_merton_pdi_put()
print("Down-and-In Put (PDI):", pdi_put)

Down-and-In Put (PDI): 2.74


In [ ]:
print("European Put (λ = 0.75): $", put8)
print("Down-and-In Put (PDI): $", pdi_put)

European Put (λ = 0.75): $ 7.2
Down-and-In Put (PDI): $ 2.74
